# 10.6 K-Means – Exercises

## Exercise 1: Old Faithful geyser – eruption duration vs. waiting time
Dataset: seaborn's built-in `geyser` (Old Faithful, Yellowstone) – loads from seaborn's data repository in Colab.

In [ ]:
import seaborn as sns, pandas as pd, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

geyser = sns.load_dataset('geyser')
print(geyser.shape)
geyser.head()

In [ ]:
X = geyser[['duration', 'waiting']]
Xs = StandardScaler().fit_transform(X)
wcss = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs).inertia_ for k in range(1, 9)]
plt.plot(range(1, 9), wcss, 'o-'); plt.xlabel('k'); plt.ylabel('WCSS'); plt.title('Elbow method'); plt.show()

In [ ]:
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Xs)
geyser['cluster'] = km.labels_
centers = StandardScaler().fit(X).inverse_transform(km.cluster_centers_)
plt.scatter(geyser.waiting, geyser.duration, c=geyser.cluster, cmap='coolwarm')
plt.scatter(centers[:,1], centers[:,0], c='black', s=200, marker='X', label='centroids')
plt.xlabel('Waiting time between eruptions (min)'); plt.ylabel('Eruption duration (min)')
plt.title('Old Faithful clusters'); plt.legend(); plt.show()
print(geyser.groupby('cluster')[['duration', 'waiting']].mean())

## Exercise 2: Image compression (396 x 396 x 3) with K-Means
Uses scikit-learn's bundled sample image, resized to 396 x 396. To use your own, replace with `Image.open('your.jpg').convert('RGB').resize((396, 396))`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from sklearn.datasets import load_sample_image
from sklearn.cluster import KMeans

img = Image.fromarray(load_sample_image('china.jpg')).resize((396, 396))
img = np.array(img)
print(img.shape)
pixels = img.reshape(-1, 3) / 255.0

In [ ]:
k = 16
km = KMeans(n_clusters=k, n_init=3, random_state=0).fit(pixels)
compressed = (km.cluster_centers_[km.labels_] * 255).astype(np.uint8).reshape(396, 396, 3)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img); ax[0].set_title('Original (16.7M colours)'); ax[0].axis('off')
ax[1].imshow(compressed); ax[1].set_title(f'K-Means, {k} colours'); ax[1].axis('off')
plt.show()
print('Original bits/pixel: 24   Compressed bits/pixel:', int(np.log2(k)))